# EXP18: ORB 策略深度分析

ORB 目前贡献不大，深入分析其表现特征。

1. ORB 独立表现（禁用 Keltner）
2. ORB 按年份/时段/方向分解
3. ORB 对整体组合的边际贡献

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, C12_KWARGS
from backtest.stats import print_ranked
import config
import pandas as pd
import numpy as np

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}
print('Data loaded')

## Part 1: 组合 vs 纯 KC vs 纯 ORB

In [ ]:
# 保存原始设置
orig_kc = config.STRATEGIES['keltner']['enabled']
orig_orb = config.STRATEGIES['orb']['enabled']

# 全部启用 (baseline)
combo = run_variant(data, "KC+ORB", **LIVE_KWARGS)

# 纯 KC
config.STRATEGIES['orb']['enabled'] = False
kc_only = run_variant(data, "KC_only", **LIVE_KWARGS)
config.STRATEGIES['orb']['enabled'] = True

# 纯 ORB
config.STRATEGIES['keltner']['enabled'] = False
orb_only = run_variant(data, "ORB_only", **LIVE_KWARGS)
config.STRATEGIES['keltner']['enabled'] = True

# 恢复
config.STRATEGIES['keltner']['enabled'] = orig_kc
config.STRATEGIES['orb']['enabled'] = orig_orb

print(f"KC+ORB: Sharpe={combo['sharpe']:.2f}, PnL=${combo['total_pnl']:.0f}, N={combo['n']}")
print(f"KC only: Sharpe={kc_only['sharpe']:.2f}, PnL=${kc_only['total_pnl']:.0f}, N={kc_only['n']}")
print(f"ORB only: Sharpe={orb_only['sharpe']:.2f}, PnL=${orb_only['total_pnl']:.0f}, N={orb_only['n']}")
print(f"\nORB marginal: Sharpe delta={combo['sharpe']-kc_only['sharpe']:+.2f}, "
      f"PnL delta=${combo['total_pnl']-kc_only['total_pnl']:+.0f}")

## Part 2: ORB 逐年分析

In [ ]:
orb_trades = [t for t in combo['_trades'] if t.strategy == 'orb']
kc_trades = [t for t in combo['_trades'] if t.strategy == 'keltner']

print(f"ORB trades: {len(orb_trades)}, KC trades: {len(kc_trades)}")

df_orb = pd.DataFrame([{
    'year': t.entry_time.year,
    'pnl': t.pnl,
    'direction': t.direction,
    'exit_reason': t.exit_reason,
    'bars_held': t.bars_held,
    'win': 1 if t.pnl > 0 else 0,
} for t in orb_trades])

if len(df_orb) > 0:
    print("\n=== ORB by Year ===")
    yearly = df_orb.groupby('year').agg(
        count=('pnl', 'count'), total_pnl=('pnl', 'sum'),
        avg_pnl=('pnl', 'mean'), win_rate=('win', 'mean'),
    ).round(2)
    print(yearly)

    print("\n=== ORB by Direction ===")
    print(df_orb.groupby('direction').agg(
        count=('pnl', 'count'), total_pnl=('pnl', 'sum'),
        avg_pnl=('pnl', 'mean'), win_rate=('win', 'mean'),
    ).round(2))

    print("\n=== ORB Exit Reasons ===")
    print(df_orb.groupby('exit_reason').agg(
        count=('pnl', 'count'), total_pnl=('pnl', 'sum'), avg_pnl=('pnl', 'mean'),
    ).round(2))
else:
    print("No ORB trades!")

## Part 3: KC 与 ORB 相关性

In [ ]:
# 按月聚合 PnL，看 KC 和 ORB 是否互补
all_trades = combo['_trades']
df_all = pd.DataFrame([{
    'month': t.entry_time.strftime('%Y-%m'),
    'pnl': t.pnl,
    'strategy': t.strategy,
} for t in all_trades])

monthly = df_all.pivot_table(index='month', columns='strategy', values='pnl', aggfunc='sum').fillna(0)
if 'keltner' in monthly.columns and 'orb' in monthly.columns:
    corr = monthly['keltner'].corr(monthly['orb'])
    print(f"Monthly PnL correlation (KC vs ORB): {corr:.3f}")
    print(f"Negative correlation = good diversification")
    
    # KC 亏损月中 ORB 表现
    kc_loss_months = monthly[monthly['keltner'] < 0]
    if len(kc_loss_months) > 0:
        orb_in_kc_loss = kc_loss_months['orb'].sum()
        print(f"\nIn {len(kc_loss_months)} months where KC lost: ORB contributed ${orb_in_kc_loss:.0f}")